#Spark Optimization SEssion -03

#✅ 1. Is the Driver Node Also a Worker?
🚫 By default, the driver node is NOT a worker node — it does not run executors, especially in YARN cluster mode, Kubernetes, or standalone with external worker nodes.
But you can allow the driver node to also run executors — this is called co-locating executors on the driver — though it’s not recommended for production clusters.

#✅ 2. Driver Node: Responsibilities
| Role                | Description                                          |
| ------------------- | ---------------------------------------------------- |
| 🧠 Coordinator      | Sends tasks to executors                             |
| 🧮 DAG Scheduler    | Builds and optimizes the execution plan              |
| 📦 Metadata Manager | Tracks RDD/DataFrame lineage                         |
| 💬 Communicator     | Talks to cluster manager (YARN/Mesos/K8s/Standalone) |
| 🧾 Collector        | Gathers result/output from executors                 |


#📝 It doesn’t run actual task computation — that’s what executors on worker nodes do.

#✅ 3. How Driver Is Launched
It depends on the deploy mode:

| Mode           | Where Driver Runs                 | Executors                                 |
| -------------- | --------------------------------- | ----------------------------------------- |
| `client` mode  | On your local machine (edge node) | On the cluster                            |
| `cluster` mode | On a node in the cluster          | On other nodes (usually excluding driver) |



#🎯 Example: Spark Submit in Cluster Mode

spark-submit \
  --deploy-mode cluster \
  --master yarn \
  --num-executors 27 \
  --executor-cores 4 \
  --executor-memory 16g \
  --driver-memory 4g


#Here, the driver runs on one of the cluster nodes, but does not run an executor by default.

#✅ 4. Can Driver Node Be Used as Worker (Run Executors)?
Yes — but only if you configure it.

In Spark Standalone mode, you can allow the driver node to register as a worker by setting:

#spark.deploy.recoveryMode=ZOOKEEPER
spark.master=local[*]  # If running in local mode


In YARN mode, if you want the driver node to also run executors:

You must not blacklist it

It must not be overwhelmed by driver tasks

#✅ 5. Realistic Best Practice in Production

| Aspect               | Recommendation                                   |
| -------------------- | ------------------------------------------------ |
| Driver Node          | Dedicated                                        |
| Executors on Driver? | ❌ Avoid (risk of memory contention)              |
| Memory               | Give 4–8 GB driver memory (`--driver-memory 4g`) |
| CPU                  | 2–4 cores is usually enough                      |
| Driver Placement     | On edge node or head node                        |
| High Availability?   | Use Spark Standalone + Zookeeper or K8s HA       |


#✅ 6. In Your 10-Node Cluster Scenario

| Node Role            | Node Count | Description                            |
| -------------------- | ---------- | -------------------------------------- |
| Driver Node (Master) | 1          | No executors; only coordinates the job |
| Worker Nodes         | 9          | Run all 27 executors (3 per node)      |


Unless you explicitly allow it, the driver node won’t host executors.

But if you're in a budget-constrained setup (e.g., development/testing):

You can use the driver node as a worker

But you must reduce the driver's workload or memory load

#✅ 7. Diagram: Driver vs. Executors

               +----------------------+
               |     Driver Node      |
               |  (no executors here) |
               +----------+-----------+
                          |
        -----------------------------------------------
        |            |             |                  |
   +---------+  +---------+   +---------+       ...  +---------+
   | Worker 1|  | Worker 2|   | Worker 3|            | Worker 9|
   | 3 Execs |  | 3 Execs |   | 3 Execs |            | 3 Execs |
   +---------+  +---------+   +---------+            +---------+


#✅ Summary

| Topic                   | Detail                                            |
| ----------------------- | ------------------------------------------------- |
| Driver runs tasks?      | ❌ No (only coordinates)                           |
| Can driver be a worker? | ✅ Yes (but avoid in prod)                         |
| Memory recommended      | 4–8 GB                                            |
| Deploy mode impact      | In `cluster` mode, driver runs inside the cluster |
| Default setup           | Driver node not used for executors                |


In [0]:
spark.version

In [0]:
%sql
SELECT version();

In [0]:
orders_df = spark.read.format("csv")\
            .option("header",True)\
            .load("/Volumes/datasets/practice/internal_files/order_data.csv")

In [0]:
orders_df.display()

In [0]:
orders_df.rdd.getNumPartitions()

In [0]:
orders_df.explain(True)

In [0]:
len(orders_df.inputFiles())

| Stage                  | Purpose                                          | Example Insight                      |
| ---------------------- | ------------------------------------------------ | ------------------------------------ |
| Parsed Logical Plan    | Initial syntax tree                              | Captures your command                |
| Analyzed Logical Plan  | Validates column names & types                   | Confirms schema                      |
| Optimized Logical Plan | Applies logical rules (filter pushdown, pruning) | Optimizes query plan                 |
| Physical Plan          | Actual execution plan                            | Shows file scan, filters, partitions |


✅ When to Use explain(True)
To debug slow Spark queries

To verify column pruning, predicate pushdown

To see if broadcast joins are applied

To monitor number of partitions or shuffle

In [0]:
orders_df.select("Country").display()

In [0]:
orders_df.queryExecution()

In [0]:
orders_df.printSchema()

In [0]:
orders_df.display(10)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
# orders_df_1 = orders_df.withColumn("InvoiceNo", col("InvoiceNo").cast(Int_type()))
orders_df_1 = orders_df.withColumn("Quantity", col("Quantity").cast("int"))                         

In [0]:
orders_df_1.display()

In [0]:
orders_df_1.head()

In [0]:
orders_df_1.tail(10)